In [ ]:
from google.colab import drive
drive.mount('/mnt/drive', force_remount=True)


Mounted at /mnt/drive


In [ ]:
import os
import time
import json
import zipfile
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments, TrainerCallback

drive_path = "/mnt/drive/MyDrive/flan_t5_squad_project"

# رقم chunk السابق والحالي
PREV_CHUNK_ID = 1 
CURRENT_CHUNK_ID = 2اً

# مسار النموذج السابق 
prev_model_path_or_zip = os.path.join(drive_path, f"chunk_{PREV_CHUNK_ID}")
prev_zip_path = prev_model_path_or_zip + ".zip"

# فك الضغط 
if prev_model_path_or_zip.endswith(".zip") or os.path.exists(prev_zip_path):
    if os.path.exists(prev_zip_path):
        if not os.path.exists(prev_model_path_or_zip):
            print(f" جاري فك ضغط النموذج السابق chunk_{PREV_CHUNK_ID} ...")
            with zipfile.ZipFile(prev_zip_path, "r") as zip_ref:
                zip_ref.extractall(prev_model_path_or_zip)
            print(" تم فك الضغط!")
    model_path = prev_model_path_or_zip
else:
    model_path = prev_model_path_or_zip

# تأكيد وجود المجلد المفكوك
if not os.path.exists(model_path):
    raise FileNotFoundError(f" غير موجود (لمجلد المفكوك للنموذج السابق): {model_path}")

print(f" تحميل النموذج السابق من: {model_path}")
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path, local_files_only=True)
print(" النموذج السابق جاهز")

# تحميل البيانات
train_file = "/mnt/drive/MyDrive/flan_t5_project/train-v1.1.json"
with open(train_file, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

contexts, questions = [], []
for article in raw_data["data"]:
    for paragraph in article["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            contexts.append(context)
            questions.append(qa["question"])

dataset = Dataset.from_dict({"context": contexts, "question": questions})
train_valid = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset, val_dataset = train_valid["train"], train_valid["test"]

chunk_count = 5
chunk_size = len(train_dataset) // chunk_count
chunks = []
for i in range(chunk_count):
    start_idx = i * chunk_size
    end_idx = len(train_dataset) if i == chunk_count - 1 else (i + 1) * chunk_size
    chunks.append(train_dataset.select(range(start_idx, end_idx)))

def preprocess_function(examples):
    inputs = [f"generate question: {c}" for c in examples["context"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(examples["question"], max_length=64, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = chunks[CURRENT_CHUNK_ID].map(preprocess_function, batched=True, remove_columns=["context", "question"])
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=["context", "question"])

output_dir = os.path.join(drive_path, f"chunk_{CURRENT_CHUNK_ID}")
os.makedirs(output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
    save_total_limit=2,
)

class EpochPrinter(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        print(f" انتهى Epoch رقم {int(state.epoch)}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    callbacks=[EpochPrinter()],
)

start = time.time()
trainer.train()
end = time.time()

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

zip_output = output_dir + ".zip"
with zipfile.ZipFile(zip_output, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start=output_dir)
            zipf.write(file_path, arcname)

print(f" تم تدريب chunk_{CURRENT_CHUNK_ID} وحفظه وضغطه بنجاح!")
print(f" الوقت المستغرق: {end - start:.2f} ثانية")

import sys
sys.exit(f" توقف التدريب عند chunk_{CURRENT_CHUNK_ID}. شغّلي السكربت مرة ثانية مع CURRENT_CHUNK_ID = {CURRENT_CHUNK_ID + 1}")


📦 تحميل النموذج السابق من: /mnt/drive/MyDrive/flan_t5_squad_project/chunk_1
✅ النموذج السابق جاهز


Map:   0%|          | 0/15767 [00:00<?, ? examples/s]

Map:   0%|          | 0/8760 [00:00<?, ? examples/s]

/tmp/ipython-input-6-1472321762.py:94: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.480500,0.425912
2,0.468000,0.424364
3,0.460900,0.424451


✅ انتهى Epoch رقم 1
✅ انتهى Epoch رقم 2
✅ انتهى Epoch رقم 3
✅ تم تدريب chunk_2 وحفظه وضغطه بنجاح!
🕒 الوقت المستغرق: 2418.46 ثانية


SystemExit: ✅ توقف التدريب عند chunk_2. شغّلي السكربت مرة ثانية مع CURRENT_CHUNK_ID = 3

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import os
import time
import json
import zipfile
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments, TrainerCallback

drive_path = "/mnt/drive/MyDrive/flan_t5_squad_project"

# رقم chunk السابق والحالي
PREV_CHUNK_ID = 2 
CURRENT_CHUNK_ID = 3  

# مسار النموذج السابق 
prev_model_path_or_zip = os.path.join(drive_path, f"chunk_{PREV_CHUNK_ID}")
prev_zip_path = prev_model_path_or_zip + ".zip"

# فك الضغط  zip
if prev_model_path_or_zip.endswith(".zip") or os.path.exists(prev_zip_path):
    if os.path.exists(prev_zip_path):
        if not os.path.exists(prev_model_path_or_zip):
            print(f" جاري فك ضغط النموذج السابق chunk_{PREV_CHUNK_ID} ...")
            with zipfile.ZipFile(prev_zip_path, "r") as zip_ref:
                zip_ref.extractall(prev_model_path_or_zip)
            print(" تم فك الضغط!")
    model_path = prev_model_path_or_zip
else:
    model_path = prev_model_path_or_zip

# تأكيد وجود المجلد المفكوك
if not os.path.exists(model_path):
    raise FileNotFoundError(f"لم أجد المجلد المفكوك للنموذج السابق: {model_path}")

print(f" تحميل النموذج السابق من: {model_path}")
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path, local_files_only=True)
print(" النموذج السابق جاهز")

# تحميل البيانات
train_file = "/mnt/drive/MyDrive/flan_t5_project/train-v1.1.json"
with open(train_file, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

contexts, questions = [], []
for article in raw_data["data"]:
    for paragraph in article["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            contexts.append(context)
            questions.append(qa["question"])

dataset = Dataset.from_dict({"context": contexts, "question": questions})
train_valid = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset, val_dataset = train_valid["train"], train_valid["test"]

chunk_count = 5
chunk_size = len(train_dataset) // chunk_count
chunks = []
for i in range(chunk_count):
    start_idx = i * chunk_size
    end_idx = len(train_dataset) if i == chunk_count - 1 else (i + 1) * chunk_size
    chunks.append(train_dataset.select(range(start_idx, end_idx)))

def preprocess_function(examples):
    inputs = [f"generate question: {c}" for c in examples["context"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(examples["question"], max_length=64, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = chunks[CURRENT_CHUNK_ID].map(preprocess_function, batched=True, remove_columns=["context", "question"])
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=["context", "question"])

output_dir = os.path.join(drive_path, f"chunk_{CURRENT_CHUNK_ID}")
os.makedirs(output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
    save_total_limit=2,
)

class EpochPrinter(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        print(f" انتهى Epoch رقم {int(state.epoch)}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    callbacks=[EpochPrinter()],
)

start = time.time()
trainer.train()
end = time.time()

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

zip_output = output_dir + ".zip"
with zipfile.ZipFile(zip_output, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start=output_dir)
            zipf.write(file_path, arcname)

print(f" تم تدريب chunk_{CURRENT_CHUNK_ID} وحفظه وضغطه بنجاح!")
print(f" الوقت المستغرق: {end - start:.2f} ثانية")

import sys
sys.exit(f" توقف التدريب عند chunk_{CURRENT_CHUNK_ID}. شغّلي السكربت مرة ثانية مع CURRENT_CHUNK_ID = {CURRENT_CHUNK_ID + 1}")


📦 تحميل النموذج السابق من: /mnt/drive/MyDrive/flan_t5_squad_project/chunk_2
✅ النموذج السابق جاهز


Map:   0%|          | 0/15767 [00:00<?, ? examples/s]

Map:   0%|          | 0/8760 [00:00<?, ? examples/s]

/tmp/ipython-input-7-2506536480.py:94: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,0.476400,0.422327
2,0.463600,0.421278
3,0.457800,0.421185


✅ انتهى Epoch رقم 1
✅ انتهى Epoch رقم 2
✅ انتهى Epoch رقم 3
✅ تم تدريب chunk_3 وحفظه وضغطه بنجاح!
🕒 الوقت المستغرق: 2427.62 ثانية


SystemExit: ✅ توقف التدريب عند chunk_3. شغّلي السكربت مرة ثانية مع CURRENT_CHUNK_ID = 4

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [5]:
c

 تحميل النموذج السابق من: /mnt/drive/MyDrive/flan_t5_squad_project/chunk_3
 النموذج السابق جاهز


Map:   0%|          | 0/15771 [00:00<?, ? examples/s]

Map:   0%|          | 0/8760 [00:00<?, ? examples/s]

/tmp/ipython-input-5-3434982980.py:94: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.473900,0.419633
2,0.462600,0.418996
3,0.453800,0.417892
4,0.447800,0.417781
5,0.444600,0.418362


 انتهى Epoch رقم 1
 انتهى Epoch رقم 2
 انتهى Epoch رقم 3
 انتهى Epoch رقم 4
 انتهى Epoch رقم 5
 تم تدريب chunk_4 وحفظه وضغطه بنجاح!
 الوقت المستغرق: 3971.06 ثانية


SystemExit:  توقف التدريب عند chunk_4. شغّلي السكربت مرة ثانية مع CURRENT_CHUNK_ID = 5

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
